# Student v2 training
This experiment tests a 16-channel, four-block student against the current eight-channel, three-block model. Enable a GPU in **Runtime > Change runtime type**. Upload the supplied `realtime-image-sharpening-v2-colab.zip` archive. Run cells in order. The original checkpoints stay unchanged.


In [ ]:
from google.colab import files
from pathlib import Path
from zipfile import ZipFile
uploaded = files.upload()
archive = Path(next(iter(uploaded)))
with ZipFile(archive) as source:
    source.extractall('/content')
project = Path('/content/realtime-image-sharpening')
assert (project / 'train.py').exists()
%cd /content/realtime-image-sharpening


In [ ]:
requirements = project / 'requirements-training.txt'
assert requirements.exists(), f'Missing {requirements}; rerun the upload cell.'
!pip -q install -r /content/realtime-image-sharpening/requirements-training.txt
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime and rerun this cell.'
print('Training GPU:', torch.cuda.get_device_name(0))


The official DIV2K training archive downloads inside Colab. It is separate from the 125-image test set on your laptop. This download requires several GB of free space.


In [ ]:
!python download_data.py --split train


Train both variants with identical settings. Twelve epochs is an initial experiment. Save the output ZIP before ending the Colab runtime. If Colab disconnects, local files can be lost.


In [ ]:
!python train.py --mode baseline --epochs 12 --channels 16 --blocks 4 --batch-size 4 --patch-size 256 --output-dir checkpoints_v2


In [ ]:
!python train.py --mode distill --epochs 12 --channels 16 --blocks 4 --batch-size 4 --patch-size 256 --output-dir checkpoints_v2


In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
names = ['baseline_best.pt', 'distill_best.pt', 'baseline_history.csv', 'distill_history.csv']
with ZipFile('/content/student_v2_checkpoints.zip', 'w', ZIP_DEFLATED) as output:
    for name in names:
        source = project / 'checkpoints_v2' / name
        assert source.exists(), f'Missing {source}'
        output.write(source, arcname=name)
files.download('/content/student_v2_checkpoints.zip')
